# Trading Bot Training - Colab Pro+

**Fixed training configuration**

In [ ]:
# 1. Check GPU
!nvidia-smi
import torch
print(f"\nCUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Install dependencies
!pip install -q pybit
print("Done!")

In [ ]:
# 3. Mount Drive and extract project
from google.colab import drive
import zipfile, os

drive.mount('/content/drive')

with zipfile.ZipFile('/content/drive/MyDrive/trading_bot.zip', 'r') as z:
    z.extractall('/content')

os.chdir('/content/trading_bot')
print(f"Ready: {os.getcwd()}")

In [ ]:
# 4. Fetch data from Bybit
import numpy as np
import pandas as pd
from pybit.unified_trading import HTTP
import time

print("Fetching 60 days of 5m candles...")
session = HTTP(testnet=False)
symbols = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']

all_data = []
for symbol in symbols:
    klines = []
    end_time = None
    target = 17280  # ~60 days
    while len(klines) < target:
        params = {'category': 'linear', 'symbol': symbol, 'interval': '5', 'limit': 1000}
        if end_time: params['end'] = end_time
        resp = session.get_kline(**params)
        if resp['retCode'] != 0 or not resp['result']['list']: break
        batch = resp['result']['list']
        klines.extend(batch)
        end_time = int(batch[-1][0]) - 1
        time.sleep(0.05)
    
    df = pd.DataFrame(klines, columns=['timestamp','open','high','low','close','volume','turnover'])
    df = df.astype({'open':float,'high':float,'low':float,'close':float,'volume':float})
    df = df.drop_duplicates('timestamp').sort_values('timestamp').reset_index(drop=True)
    
    # Calculate features
    df['returns'] = df['close'].pct_change()
    df['volatility'] = df['returns'].rolling(20).std()
    df['rsi'] = 100 - 100/(1 + df['close'].diff().clip(lower=0).rolling(14).mean() / (df['close'].diff().clip(upper=0).abs().rolling(14).mean() + 1e-8))
    df['macd'] = df['close'].ewm(span=12).mean() - df['close'].ewm(span=26).mean()
    df['macd_signal'] = df['macd'].ewm(span=9).mean()
    df['macd_hist'] = df['macd'] - df['macd_signal']
    df['bb_mid'] = df['close'].rolling(20).mean()
    df['bb_std'] = df['close'].rolling(20).std()
    df['bb_position'] = (df['close'] - (df['bb_mid'] - 2*df['bb_std'])) / (4*df['bb_std'] + 1e-8)
    df['bb_width'] = 4*df['bb_std'] / (df['bb_mid'] + 1e-8)
    df['atr'] = pd.concat([df['high']-df['low'], (df['high']-df['close'].shift()).abs(), (df['low']-df['close'].shift()).abs()], axis=1).max(axis=1).rolling(14).mean()
    df['atr_pct'] = df['atr'] / df['close']
    df['momentum_3'] = df['close'].pct_change(3)
    df['momentum_6'] = df['close'].pct_change(6)
    df['momentum_12'] = df['close'].pct_change(12)
    df['volume_ratio'] = df['volume'] / (df['volume'].rolling(20).mean() + 1e-8)
    df['price_sma10_ratio'] = df['close'] / df['close'].rolling(10).mean() - 1
    df['price_sma20_ratio'] = df['close'] / df['close'].rolling(20).mean() - 1
    
    all_data.append(df.dropna())
    print(f"{symbol}: {len(df.dropna())} candles")

combined = pd.concat(all_data, ignore_index=True)
print(f"\nTotal: {len(combined)} samples")

In [ ]:
# 5. Prepare features with proper normalization
feature_cols = ['close','returns','volatility','rsi','macd','macd_signal','macd_hist',
                'bb_position','bb_width','atr_pct','momentum_3','momentum_6','momentum_12',
                'volume_ratio','price_sma10_ratio','price_sma20_ratio']

# Store normalization params for later use
norm_params = {}
for col in feature_cols[1:]:  # Skip 'close'
    m, s = combined[col].mean(), combined[col].std()
    norm_params[col] = {'mean': m, 'std': s}
    if s > 0: 
        combined[col] = (combined[col] - m) / s

# Also normalize close price using log returns
combined['close_norm'] = np.log(combined['close'] / combined['close'].shift(1)).fillna(0)
combined['close_norm'] = combined['close_norm'] / (combined['close_norm'].std() + 1e-8)

# Use normalized close instead of raw close
feature_cols_norm = ['close_norm'] + feature_cols[1:]
features = combined[feature_cols_norm].values

# Pad to 32 features
if features.shape[1] < 32:
    pad = np.zeros((len(features), 32 - features.shape[1]))
    for i in range(pad.shape[1]):
        window = [3, 6, 12, 24, 48][i % 5]
        ret = combined['close'].pct_change(window).fillna(0).values
        std = np.std(ret)
        if std > 0:
            ret = ret / std
        pad[:, i] = ret
    features = np.hstack([features, pad])

data = features.astype(np.float32)
print(f"Data shape: {data.shape}")
print(f"Feature stats - Mean: {data.mean():.4f}, Std: {data.std():.4f}, Min: {data.min():.4f}, Max: {data.max():.4f}")

In [ ]:
# 6. Direct training (skip the trainer class for better control)
import sys
sys.path.insert(0, '/content/trading_bot')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from models.transformer_gru import TransformerGRU, TransformerGRUConfig
from pathlib import Path

device = 'cuda'
print(f"Device: {device}")

# Create sequences and labels
seq_len = 48
pred_horizon = 3
threshold = 0.001  # 0.1% move threshold

sequences = []
labels = []

# Use raw close for labels (before normalization)
raw_close = combined['close'].values

for i in range(seq_len, len(data) - pred_horizon):
    seq = data[i-seq_len:i]
    
    # Calculate future return for label
    current_price = raw_close[i]
    future_price = raw_close[i + pred_horizon]
    ret = (future_price - current_price) / current_price
    
    # 3-class label: 0=down, 1=neutral, 2=up
    if ret < -threshold:
        label = 0
    elif ret > threshold:
        label = 2
    else:
        label = 1
    
    sequences.append(seq)
    labels.append(label)

X = np.array(sequences, dtype=np.float32)
y = np.array(labels, dtype=np.int64)

print(f"Sequences: {X.shape}")
print(f"Class distribution: Down={np.mean(y==0):.1%}, Neutral={np.mean(y==1):.1%}, Up={np.mean(y==2):.1%}")

# Split data
n = len(X)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

In [ ]:
# 7. Create model and train
model_config = TransformerGRUConfig(
    input_dim=32,
    d_model=256,
    n_heads=8,
    n_encoder_layers=3,
    d_ff=1024,
    dropout=0.2,  # Slightly higher dropout
    gru_hidden=256,
    output_dim=3
)

model = TransformerGRU(model_config).to(device)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

# Class weights to handle imbalance
class_counts = np.bincount(y_train)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * 3  # Normalize
class_weights = torch.FloatTensor(class_weights).to(device)
print(f"Class weights: {class_weights.cpu().numpy()}")

# Loss with class weights and label smoothing
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

# Optimizer with lower LR
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

# Scheduler with warmup
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, 
    max_lr=3e-4,
    epochs=100,
    steps_per_epoch=len(X_train) // 64 + 1,
    pct_start=0.1,  # 10% warmup
    anneal_strategy='cos'
)

# Data loaders
train_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train)),
    batch_size=64, shuffle=True
)
val_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y_val)),
    batch_size=128
)
test_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test)),
    batch_size=128
)

In [ ]:
# 8. Training loop with logging
import os
os.makedirs('/content/checkpoints', exist_ok=True)

best_val_acc = 0
best_epoch = 0
patience = 20
patience_counter = 0

for epoch in range(100):
    # Train
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output['direction'], batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        train_loss += loss.item()
        pred = output['direction'].argmax(dim=-1)
        train_correct += (pred == batch_y).sum().item()
        train_total += batch_y.size(0)
    
    # Validate
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            output = model(batch_x)
            loss = criterion(output['direction'], batch_y)
            val_loss += loss.item()
            pred = output['direction'].argmax(dim=-1)
            val_correct += (pred == batch_y).sum().item()
            val_total += batch_y.size(0)
    
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    
    # Log every epoch
    print(f"Epoch {epoch:3d}: Train Loss={avg_train_loss:.4f}, Acc={train_acc:.4f} | Val Loss={avg_val_loss:.4f}, Acc={val_acc:.4f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        patience_counter = 0
        torch.save({
            'model': model.state_dict(),
            'epoch': epoch,
            'val_loss': avg_val_loss,
            'val_acc': val_acc,
            'config': model_config
        }, '/content/checkpoints/transformer_best.pt')
        print(f"  -> Saved best model!")
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch}")
        break

print(f"\nBest model: Epoch {best_epoch}, Val Acc={best_val_acc:.4f}")

In [ ]:
# 9. Test evaluation
checkpoint = torch.load('/content/checkpoints/transformer_best.pt')
model.load_state_dict(checkpoint['model'])
model.eval()

test_correct = 0
test_total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        output = model(batch_x)
        pred = output['direction'].argmax(dim=-1)
        test_correct += (pred == batch_y).sum().item()
        test_total += batch_y.size(0)
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())

test_acc = test_correct / test_total
print(f"\nTest Accuracy: {test_acc:.4f}")

# Confusion matrix
from collections import Counter
pred_counts = Counter(all_preds)
print(f"\nPrediction distribution:")
print(f"  Down: {pred_counts[0]} ({pred_counts[0]/len(all_preds):.1%})")
print(f"  Neutral: {pred_counts[1]} ({pred_counts[1]/len(all_preds):.1%})")
print(f"  Up: {pred_counts[2]} ({pred_counts[2]/len(all_preds):.1%})")

In [ ]:
# 10. Save to Drive and download
import shutil
import json

# Save results
results = {
    'best_epoch': int(best_epoch),
    'best_val_acc': float(best_val_acc),
    'test_acc': float(test_acc),
    'model_config': {
        'd_model': 256,
        'n_layers': 3,
        'n_heads': 8,
        'dropout': 0.2
    }
}

with open('/content/training_results.json', 'w') as f:
    json.dump(results, f, indent=2)

# Copy to Drive
!cp /content/checkpoints/transformer_best.pt /content/drive/MyDrive/
!cp /content/training_results.json /content/drive/MyDrive/

print("Saved to Google Drive!")
print(f"  - transformer_best.pt")
print(f"  - training_results.json")